# CISA Known Exploited Vulnerabilities (KEV) Neural Networks
Data source: The Cybersecurity and Infrastructure Security Agency's (CISA) [Known Exploited Vulnerabilities (KEV) catalog](https://github.com/cisagov/kev-data?)

## Background
**Exploratory Analysis**
<br>
I previously performed an exploratory data analysis on the KEV dataset to identify trends in known exploited vulnerabilities, remediation-timelines, and ransomware-associated weaknesses. Find my exploratory analysis [here](https://github.com/ClarenceBallensky/CISA_KEV_Analysis).

My findings were as follows:
- Microsoft has the largest number of cataloged vulnerabilities.
- Input validation and use-after-free are the most common weaknesses.
- Ransomware vulnerabilities show similar remediation timelines to the broader catalog.
- Most remediation deadlines are exactly 21 days.
- Vulnerability additions peaked in 2022.

**Machine Learning**
<br>
I also built several machine learning models to classify ransomware status--including a logistic regression, decision forest, and support vector machine model. I then compared performance. 
Find my machine learning models [here](https://github.com/ClarenceBallensky/CISA_KEV_Machine-Learning).

My models ranked as follows:
1) logistic regression
2) random forest
3) support vector machine

My best model, logistic regression, produced the following scores
<br>
Accuracy score: 0.8
<br>
Precision score: 0.5016611295681063
<br>
Recall score: 0.45481927710843373
<br>
f1 score: 0.47709320695102686
<br>

Note that accuracy alone is not a meaningful benchmark here, since a model that always predicts 'Unknown' would already score ~80%. The logistic regression model's real gains were in precision and recall--catching a meaningfully higher share of actual ransomware-associated CVEs than a naive baseline would.

## Step 1: Load the Data

In [1]:
import pandas as pd

# Get the .csv data from GitHub
data_url = "https://raw.githubusercontent.com/cisagov/kev-data/refs/heads/develop/known_exploited_vulnerabilities.csv"
kev = pd.read_csv(data_url)

# Save a local copy
kev.to_csv("known_exploited_vulnerabilities.csv", index=False)
kev = pd.read_csv("known_exploited_vulnerabilities.csv")

print(kev.head(30))

             cveID    vendorProject  \
0   CVE-2026-20349            Cisco   
1   CVE-2026-68820        Microsoft   
2   CVE-2026-72898         Metabase   
3    CVE-2026-8037         Progress   
4   CVE-2026-63077        JetBrains   
5   CVE-2026-18556           N-able   
6   CVE-2026-34486           Apache   
7    CVE-2026-9198              IBM   
8   CVE-2026-18577           N-able   
9   CVE-2026-20316            Cisco   
10  CVE-2025-68686         Fortinet   
11  CVE-2026-16812           Arista   
12  CVE-2026-16232      Check Point   
13  CVE-2026-50522        Microsoft   
14  CVE-2026-60137        WordPress   
15  CVE-2026-63030        WordPress   
16   CVE-2026-0770         Langflow   
17  CVE-2021-27137           DD-WRT   
18  CVE-2026-58644        Microsoft   
19  CVE-2026-25089         Fortinet   
20  CVE-2026-39808         Fortinet   
21  CVE-2026-46817           Oracle   
22   CVE-2023-4346  KNX Association   
23  CVE-2026-56155        Microsoft   
24  CVE-2026-56164       

## Step 2: Read the Data Documentation 
### Schema
#### (Extracted from known_exploited_vulnerabilities_schema.json)

| Column | Description |
| :--- | ---: | 
| cveID | The CVE ID of the vulnerability in the format CVE-YYYY-NNNN, note that the number portion can have more than 4 digits |
| vendorProject | The vendor or project name for the vulnerability |
| product | The vulnerability product |
| vulnerabilityName | The name of the vulnerability |
| dateAdded | The date the vulnerability was added to the catalog in the format YYYY-MM-DD |
| shortDescription | A short description of the vulnerability |
| requiredAction | The required action to address the vulnerability |
| dueDate | The date the required action is due in the format YYYY-MM-DD |
| knownRansomwareCampaignUse | 'Known' if this vulnerability is known to have been leveraged as part of a ransomware campaign; 'Unknown' if CISA lacks confirmation that the vulnerability has been utilized for ransomware |
| notes | Any additional notes about the vulnerability |
| cwes | Common Weakness Enumeration (CWE) codes associated with this vulnerability. CWEs are in the format CWE-NNNN; note that the number portion can have any number of digits |


## Step 3: Inspect the Data

In [2]:
print(kev.info())
print()
print()
print(kev.describe(include="all"))
print()
print()
kev.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1665 entries, 0 to 1664
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   cveID                       1665 non-null   str  
 1   vendorProject               1665 non-null   str  
 2   product                     1665 non-null   str  
 3   vulnerabilityName           1665 non-null   str  
 4   dateAdded                   1665 non-null   str  
 5   shortDescription            1665 non-null   str  
 6   requiredAction              1665 non-null   str  
 7   dueDate                     1665 non-null   str  
 8   knownRansomwareCampaignUse  1665 non-null   str  
 9   notes                       1665 non-null   str  
 10  cwes                        1494 non-null   str  
dtypes: str(11)
memory usage: 1.0 MB
None


                 cveID vendorProject  product  \
count             1665          1665     1665   
unique            1665           276      674  

np.int64(0)

## Step 4: Clean the Data
Before creating my models, I cleaned the dataset by renaming columns to snake_case and converting date columns to datetime objects.

In [3]:
# Rename columns to match Python's snake_case convention
kev.rename(columns={
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use"
}, inplace=True)

#print(kev.columns)



# Covert values in date_added and due_date from strings to dates
kev["date_added"] = pd.to_datetime(kev["date_added"])
kev["due_date"] = pd.to_datetime(kev["due_date"])

#print(kev.info())

## Step 5: Choosing the Model

Since I will be running this model locally, I am limited by my computer's hardware; in particular, I only have access to a CPU, not a GPU. Below is an assessment of the various models I learned about in Codecademy's "Engineer Neural Networks with PyTorch and Transformers" skill path for my use case. 

| Technique | Is Appropriate? | Explanation |
| :--- | ---- | ---: |
| DistilBERT (fine-tune) | Appropriate | Distilled BERT variant (~66M parameters); retains ~97% of BERT's language understanding at a fraction of the compute cost, making it well-suited for fine-tuning on CPU hardware |
| GRU (trained from scratch) | Has potential | Fewer gates than LSTM, reducing parameter count and training time; a reasonable sequential baseline for short-text classification |
| LSTM (trained from scratch) | Has potential | Captures longer-range dependencies via its cell state, but its additional gating mechanisms increase training time relative to GRU with limited benefit on short vulnerability descriptions |
| CIFAR10-style CNN | Inappropriate | Architecture is designed for image classification; the KEV catalog contains no image data, so there is no relevant input modality |
| CLIP fine-tuning | Inappropriate | A multimodal image-text model; excluded due to the absence of an image modality in this dataset |

<br>


**I will try fine-tuning DistilBERT and then assess its performance.**

## Step 6: Process the Data

### Importing Libraries

In [4]:
# For splitting the data
from sklearn.model_selection import train_test_split

# For converting the dataframe to a dataset
from datasets import Dataset

### Splitting the Data

In [5]:
text_df = pd.DataFrame({
    "text": kev["vendor_project"] + " " + kev["product"] + ": " + kev["short_description"],
    "label": kev["known_ransomware_campaign_use"]
})

#### Standard

The value that I am predicting, `known_ransomware_campaign_use`, is imbalanced (~20% "Known", ~80% "Unknown"). I will use include the `stratify` argument in order to preserve the ~80/20 class ratio in every fold.

In [6]:
# Splitting the data into training and testing sets

train_df, test_df = train_test_split(text_df, test_size=0.2, random_state=42, stratify=text_df["label"])

In [7]:
# Convert Pandas Dataframes to Hugging Face Datasets

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

#### Chronological

A chronological split is a more realistic evaluation than random splitting for this dataset specifically: KEV entries are added over time as new vulnerabilities are discovered and exploited, and ransomware tactics/targets shift over time too. One of the weaknesses in my machine learning project on this same dataset was that none of my models accounted for time.

One real complication: class imbalance interacts with time. The stratified random split guarantees that both the train and test sets have a representative ratio of ransomware-associated CVEs. A chronological split doesn't guarantee that; if ransomware-flagged CVEs cluster in a particular time window, the test set could end up with very few (or many) positive examples.

In [8]:
text_df["date_added"] = kev["date_added"]

In [9]:
# Splitting the data into training and testing sets based on the date the vulnerability was added to the KEV dataset

text_df = text_df.sort_values("date_added").reset_index(drop=True)
split_index = int(len(text_df) * 0.8)
train_df_chron = text_df.iloc[:split_index]
test_df_chron = text_df.iloc[split_index:]

In [10]:
train_df_chron = train_df_chron.drop(columns=["date_added"])
test_df_chron = test_df_chron.drop(columns=["date_added"])

In [11]:
# Convert Pandas Dataframes to Hugging Face Datasets

train_dataset_chron = Dataset.from_pandas(train_df_chron)
test_dataset_chron = Dataset.from_pandas(test_df_chron)

## Step 7: Implementing the Model

### Importing Libraries

In [12]:
# Models
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

# For eveluating the models
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# Setting a random seed
import random
import torch
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
set_seed()

### Creating a Method to Evaluate Model Performance

The Confusion Matrix evaluates how well a classification model performs by comparing predicted outcomes against true values.

The upper lefthand corner gives the count of true positives, the lower lefthand corner gives the count of false positives, the upper righthand corner gives the count of false negatives, and the lower righthand corner gives the count of true negatives.

<br>

Accuracy measures how many classifications the algorithm got correct out of every classification it made.

Precision is the ratio of correct positive classifications to all positive classifications made by the model.

Recall is the ratio of correct positive classifications made by the model to all actual positives.

F1-score is a combination of precision and recall. The formula for the f1-score uses a harmonic mean, so it will be low if either precision or recall is low.


In [13]:
label_map = {"Unknown": 0, "Known": 1}

def evaluate_model(model, tokenizer, test_dataset, device):
    model.eval()
    all_preds = [] 
    all_labels = []

    with torch.no_grad():
        for row in test_dataset:
            inputs = tokenizer(row["text"], return_tensors="pt", truncation=True, padding=True).to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            
            all_preds.append(pred)
            all_labels.append(label_map[row["label"]])
    

    cm = confusion_matrix(all_labels, all_preds)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    print(f"Confusion matrix: \n{cm}")
    # upper lefthand corner = true positives
    # lower lefthand corner = false positives
    # upper righthand corner = false negatives
    # lower righthand corner = true negatives
    
    print()
    
    print(f"Accuracy score: {accuracy}")
    print(f"Precision score: {precision}")
    print(f"Recall score: {recall}")
    print(f"f1 score: {f1}")

### Loading the Base Model

In [14]:
device = "cpu"
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Tokenizing the Data

In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="longest", truncation=True)

In [16]:
tokenized_train_datset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

tokenized_train_datset_chron = train_dataset_chron.map(tokenize_function, batched=True)
tokenized_test_dataset_chron = test_dataset_chron.map(tokenize_function, batched=True)

Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

### Base Model Evaluation

In [17]:
evaluate_model(model, tokenizer, test_dataset, device)

Confusion matrix: 
[[261   2]
 [ 70   0]]

Accuracy score: 0.7837837837837838
Precision score: 0.0
Recall score: 0.0
f1 score: 0.0


### Fine-Tuning the Model

### Fine-Tuned Model Evaluation

## Conclusion